# Reconia — RoBERTa + bố cục 2D trên DocILE
Notebook huấn luyện đầy đủ: 4.680 train / 500 val / 500 test nội bộ, cùng split và chỉ số với GNN đã kiểm chứng.

**RUN_MODE = "train"** chạy tối đa 20 epochs. "smoke" chỉ dùng để kiểm tra kỹ thuật. Dòng MODE in rõ trước khi chạy. Mọi từ OCR được xử lý qua các cửa sổ 512 token, không bỏ phần cuối tài liệu.

GPU và Internet phải bật. Checkpoint RoBERTa-base chính thức được tải theo revision cố định; không khởi tạo backbone ngẫu nhiên. Sau khi Save & Run All hoàn tất, tải ZIP trong Outputs.


In [ ]:
RUN_MODE = "train"  # Full experiment; use "smoke" only for a technical check.
RUN_ID = "roberta-docile-seed42-v1"
EPOCHS = 20
SEED = 42
RESUME_FROM = ""  # Prior output's checkpoints directory, containing best + latest.
BACKBONE_REPO = "FacebookAI/roberta-base"
BACKBONE_REVISION = "e2da8e2f811d1448a5b465c236feacd80ffbac7b"


In [ ]:
import os, sys, json, hashlib, shutil, subprocess, zipfile, re
from pathlib import Path
assert RUN_MODE in {"smoke", "train"}
assert re.fullmatch(r"[a-zA-Z0-9_-]+", RUN_ID), "Use a simple run ID"
assert EPOCHS > 0
assert not RESUME_FROM or RUN_MODE == "train", "Resume is only supported for full training"
# Keep Kaggle's existing CUDA PyTorch, NumPy and pandas installations.
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                "python-dotenv==1.0.1", "transformers==4.57.6"], check=True)
INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
CODE = Path("/kaggle/temp/reconia-code")
DATA = Path("/kaggle/temp/reconia-docile")
def extract_bundle(filename, destination):
    matches = list(INPUT.rglob(filename))
    assert len(matches) == 1, f"Attach exactly one Reconia bundle: expected {filename}, found {matches}"
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(matches[0]) as z:
        for name in z.namelist():
            assert (destination / name).resolve().is_relative_to(destination.resolve())
        z.extractall(destination)
extract_bundle("reconia-code.bin", CODE)
extract_bundle("docile-export.bin", DATA)
provenance = json.loads((CODE / "source-provenance.json").read_text())
for name, digest in provenance["files_sha256"].items():
    assert hashlib.sha256((CODE / name).read_bytes()).hexdigest() == digest, name
os.chdir(CODE)
sys.path.insert(0, str(CODE))
os.environ["DOCILE_EXPORT_DIR"] = str(DATA)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONHASHSEED"] = str(SEED)
import torch
assert torch.cuda.is_available(), "Enable a GPU accelerator in Notebook Settings"
print("GPU:", torch.cuda.get_device_name(0), "| PyTorch:", torch.__version__)
from app.data.portable import ExportAdapter
adapter = ExportAdapter(DATA, "docile")
counts = {s: len(adapter.document_ids(s)) for s in ("train", "val", "test")}
assert counts == {"train": 4680, "val": 500, "test": 500}, counts
print("Verified dataset:", counts)
del adapter

from huggingface_hub import snapshot_download
MODEL = snapshot_download(repo_id=BACKBONE_REPO, revision=BACKBONE_REVISION,
                          cache_dir="/kaggle/temp/huggingface",
                          allow_patterns=["config.json", "model.safetensors", "tokenizer.json",
                                          "tokenizer_config.json", "vocab.json", "merges.txt"])
os.environ["ROBERTA_MODEL_PATH"] = MODEL
print("MODE:", RUN_MODE.upper(), "| Backbone:", BACKBONE_REPO, BACKBONE_REVISION)


In [ ]:
from datetime import datetime, timezone
OUT = WORK / "reconia-runs" / (RUN_ID + ("-smoke" if RUN_MODE == "smoke" else ""))
CHECKPOINTS = OUT / "checkpoints"
REPORTS = OUT / "reports"
CACHE = Path("/kaggle/temp/reconia-graphs") / OUT.name
if any(CHECKPOINTS.glob("*.pt")):
    raise RuntimeError("This RUN_ID already has checkpoints. Choose a new RUN_ID; use RESUME_FROM to continue.")
CHECKPOINTS.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)
resume_args = []
if RESUME_FROM:
    previous = Path(RESUME_FROM)
    for filename in ("roberta_docile_latest.pt", "roberta_docile_best.pt"):
        assert (previous / filename).is_file(), f"Missing resume artifact: {filename}"
        shutil.copy2(previous / filename, CHECKPOINTS / filename)
    from app.data.roberta_dataset import WINDOW_POLICY
    saved = torch.load(CHECKPOINTS / "roberta_docile_latest.pt", map_location="cpu", weights_only=False)
    assert saved.get("window_policy") == WINDOW_POLICY, "Resume requires the current window policy"
    assert saved["epoch"] < EPOCHS, "Already finished; increase EPOCHS only for a new planned schedule"
    assert saved["config"]["num_labels"] == 111
    del saved
    resume_args = ["--resume", str(CHECKPOINTS / "roberta_docile_latest.pt")]
command = [sys.executable, "-u", "-m", "app.training.train_roberta",
           "--dataset", "docile", "--device", "cuda", "--seed", str(SEED),
           "--epochs", str(EPOCHS if RUN_MODE == "train" else 1),
           "--lr", "0.00005", "--head-lr", "0.0002", "--batch-size", "4",
           "--batch-accum", "4", "--patience", "5", "--weighted-loss", "true",
           "--cache-dir", str(CACHE), "--save-dir", str(CHECKPOINTS), "--report-dir", str(REPORTS)]
if RUN_MODE == "smoke":
    command += ["--limit-train", "8", "--limit-val", "4"]
command += resume_args
configuration = {"mode": RUN_MODE, "run_id": RUN_ID, "epochs": EPOCHS, "seed": SEED,
                 "resume_from": RESUME_FROM, "counts": counts, "command": command,
                 "backbone_repo": BACKBONE_REPO, "backbone_revision": BACKBONE_REVISION,
                 "window_policy": "all-windows-first-subword-v1",
                 "gpu": torch.cuda.get_device_name(0), "started_utc": datetime.now(timezone.utc).isoformat(),
                 "source": provenance, "export_manifest_sha256": hashlib.sha256((DATA / "manifest.json").read_bytes()).hexdigest()}
(OUT / "run-config.json").write_text(json.dumps(configuration, indent=2))
(OUT / "requirements-freeze.txt").write_text(subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True))
print("Output:", OUT)
print("MODE:", RUN_MODE.upper(), "| Training documents:", 4680 if RUN_MODE == "train" else 8)
print("Launching:", " ".join(command))
# Train in a child process so stopping the notebook does not hide Python errors.
subprocess.run(command, check=True, cwd=CODE)


In [ ]:
if RUN_MODE == "smoke":
    # Reload weights in a fresh process; use validation, never test for debugging.
    subprocess.run([sys.executable, "-u", "-m", "app.evaluation.predict",
                    "--model", "roberta", "--dataset", "docile", "--split", "val",
                    "--checkpoint", str(CHECKPOINTS / "roberta_docile_best.pt"),
                    "--limit", "4", "--device", "cuda", "--report-dir", str(REPORTS)], check=True, cwd=CODE)
    print("SMOKE PASSED. For the full experiment set RUN_MODE = 'train' and Save & Run All.")
else:
    evidence = REPORTS / "predictions/predictions_roberta_docile_test.metrics.json"
    metrics = json.loads(evidence.read_text())
    assert metrics["complete"] and metrics["num_documents"] == metrics["expected_num_documents"] == 500
    print("Full local test verified:", {k: metrics[k] for k in ("precision", "recall", "f1", "protocol")})
    print("Field-value F1:", metrics["value_metrics"]["f1"])
archive = shutil.make_archive(str(WORK / OUT.name), "zip", root_dir=OUT.parent, base_dir=OUT.name)
from IPython.display import FileLink, display
print("Download and retain both best/latest checkpoints, reports and run configuration.")
os.chdir(WORK)
display(FileLink(str(Path(archive).relative_to(WORK))))


## Kết quả
- Full run phải ghi Train: 4680 / Val: 500; file test phải đủ 500 tài liệu và complete=true.
- So sánh với GNN: entity F1 66,82%, field-value F1 56,87%. Đây là điểm nội bộ, không phải điểm DocILE chính thức.
- Giữ cả best/latest checkpoint, tokenizer, cấu hình, predictions và metrics. Không dùng smoke checkpoint cho báo cáo.
- Resume: attach output đã lưu, đặt RESUME_FROM, đổi RUN_ID, giữ cấu hình và tổng EPOCHS.
